# 1\. Setup Global: Bibliotecas + Drive + Cálculo das 3 Variáveis Dependentes

In [ ]:
# Instalar bibliotecas de explicabilidade e boosting (caso não estejam ativas na sessão)
!pip install -q shap xgboost

import time
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

from sklearn.linear_model import LinearRegression, ElasticNetCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import shap

# 1. Montar o Google Drive e ler o ficheiro
drive.mount('/content/drive')
caminho_ficheiro = '/content/drive/MyDrive/A Máquina de Sondagens/sondagens_limpo_v3.xlsx'

xls = pd.ExcelFile(caminho_ficheiro)
print("Folhas disponíveis no ficheiro:", xls.sheet_names)
df = pd.read_excel(caminho_ficheiro, sheet_name=xls.sheet_names[0])

# 2. Calcular os desvios por partido e as 3 Variáveis Dependentes (por sondagem)
partidos = ['PS', 'PSD-CDS', 'BE', 'CDU', 'PAN', 'L', 'IL', 'CH']

for p in partidos:
    # Desvio absoluto em pontos percentuais: |S - R|
    df[f'desvio_abs_{p}'] = (df[p] - df[f'result_{p}']).abs()

    # Desvio percentual absoluto (APE): (|S - R| / R) * 100
    df[f'ape_{p}'] = ((df[p] - df[f'result_{p}']).abs() / df[f'result_{p}']) * 100.0

    # Medida B_p de Arzheimer & Evans (2014): |ln((S/(100-S)) / (R/(100-R)))|
    s_prop = df[p] / 100.0
    r_prop = df[f'result_{p}'] / 100.0
    df[f'bp_{p}'] = np.log((s_prop / (1.0 - s_prop)) / (r_prop / (1.0 - r_prop))).abs()

cols_abs = [f'desvio_abs_{p}' for p in partidos]
cols_ape = [f'ape_{p}' for p in partidos]
cols_bp  = [f'bp_{p}' for p in partidos]

# Médias simples por sondagem
df['Mosteller_3']   = df[cols_abs].mean(axis=1)
df['MAPE_Sondagem'] = df[cols_ape].mean(axis=1)
df['Medida_B']      = df[cols_bp].mean(axis=1)

print(f"Base de dados pronta: {df.shape[0]} sondagens e {df.shape[1]} colunas.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folhas disponíveis no ficheiro: ['limpo']
Base de dados pronta: 640 sondagens e 61 colunas.


# 2\. Estatísticas descritivas

## 2.1. Estatísticas Descritivas das Variáveis Quantitativas (Contínuas e Discretas)

In [ ]:
variaveis_quantitativas = [
    # Variáveis Dependentes (Alvo)
    'Mosteller_3', 'MAPE_Sondagem', 'Medida_B',
    # Metadados Técnicos e Operacionais (ERC)
    'amostra_obtida', 'taxa_resposta', 'n_entrevistadores',
    'n_min_entrevistas', 'n_max_entrevistas',
    # Variáveis Temporais
    'dias_ate_prox_eleicao', 'dias_desde_ulti_eleicao',
    # Estimativas de Intenção de Voto (%)
    'PS', 'PSD-CDS', 'BE', 'CDU', 'PAN', 'L', 'IL', 'CH', 'OBN'
]

tabela_quant = df[variaveis_quantitativas].agg([
    'count', 'mean', 'std', 'min',
    lambda x: x.quantile(0.25),
    'median',
    lambda x: x.quantile(0.75),
    'max'
]).T

tabela_quant.columns = ['N', 'Média', 'Desvio-Padrão', 'Mínimo', 'P25 (Q1)', 'Mediana', 'P75 (Q3)', 'Máximo']
tabela_quant['N'] = tabela_quant['N'].astype(int)
tabela_quant = tabela_quant.round(2).reset_index().rename(columns={'index': 'Variável'})

display(tabela_quant)

,Variável,N,Média,Desvio-Padrão,Mínimo,P25 (Q1),Mediana,P75 (Q3),Máximo
0,Mosteller_3,640,2.64,1.37,0.35,1.73,2.34,3.33,8.64
1,MAPE_Sondagem,640,25.45,14.12,4.81,15.57,22.27,30.85,92.81
2,Medida_B,640,0.28,0.13,0.07,0.18,0.26,0.36,0.84
3,amostra_obtida,640,842.67,357.84,0.00,605.00,802.00,1011.00,4004.00
4,taxa_resposta,640,63.36,19.25,4.88,56.03,66.12,79.60,98.82
5,n_entrevistadores,640,18.17,22.84,0.00,8.00,12.00,21.00,447.00
6,n_min_entrevistas,640,69.97,92.49,0.00,15.00,35.50,50.00,426.00
7,n_max_entrevistas,640,95.36,87.42,0.00,35.00,65.00,84.00,428.00
8,dias_ate_prox_eleicao,640,413.98,418.39,2.00,33.75,298.50,663.25,1559.00
9,dias_desde_ulti_eleicao,640,656.08,431.10,8.00,345.00,582.50,840.25,1578.00


## 2.2. Tabela de Frequências das Variáveis Categóricas e Binárias (Dummies)

In [ ]:
df['ciclo_eleitoral'] = df['data_eleicao'].dt.strftime('%Y-%m-%d')

variaveis_categoricas = [
    'instituto', 'modo_recolha', 'cliente', 'ciclo_eleitoral',
    'coligacao_PSD-CDS_na_sondagem', 'demissao_gov', 'comunic_presi'
]

lista_freq = []
for col in variaveis_categoricas:
    serie = df[col].fillna('Não especificado (NaN)')
    contagem = serie.value_counts()
    percentagem = serie.value_counts(normalize=True) * 100.0

    temp_df = pd.DataFrame({
        'Variável': col,
        'Categoria': contagem.index.astype(str),
        'Frequência (N)': contagem.values,
        'Percentagem (%)': percentagem.values.round(2)
    })
    lista_freq.append(temp_df)

tabela_cat = pd.concat(lista_freq, ignore_index=True)
display(tabela_cat)

,Variável,Categoria,Frequência (N),Percentagem (%)
0,instituto,Aximage,156,24.38
1,instituto,Eurosondagem,149,23.28
2,instituto,Intercampus,99,15.47
3,instituto,Pitagórica,91,14.22
4,instituto,CESOP-U.Católica,58,9.06
5,instituto,Marktest,34,5.31
6,instituto,ICS/ISCTE/GFK Metris,25,3.91
7,instituto,Duplimétrica/IPESPE,15,2.34
8,instituto,Consulmark2,9,1.41
9,instituto,Outro,4,0.62


## 2.3. Desagregação dos Desvios e Metadados por Ciclo Eleitoral e por Instituto

In [ ]:
# 1. Resumo por Eleição Legislativa
tabela_por_eleicao = df.groupby('ciclo_eleitoral', as_index=False).agg(
    N_Sondagens=('n_deposito', 'count'),
    Mosteller3_Media=('Mosteller_3', 'mean'),
    Mosteller3_DP=('Mosteller_3', 'std'),
    Mosteller3_Mediana=('Mosteller_3', 'median'),
    MAPE_Media=('MAPE_Sondagem', 'mean'),
    MAPE_DP=('MAPE_Sondagem', 'std'),
    MedidaB_Media=('Medida_B', 'mean'),
    MedidaB_DP=('Medida_B', 'std'),
    Amostra_Media=('amostra_obtida', 'mean'),
    TaxaResp_Media=('taxa_resposta', 'mean')
).round(2)

print("--- 1. Desvios e Metadados por Eleição Legislativa ---")
display(tabela_por_eleicao)

# 2. Resumo por Instituto de Sondagens
tabela_por_instituto = df.groupby('instituto', as_index=False).agg(
    N_Sondagens=('n_deposito', 'count'),
    Mosteller3_Media=('Mosteller_3', 'mean'),
    Mosteller3_DP=('Mosteller_3', 'std'),
    MAPE_Media=('MAPE_Sondagem', 'mean'),
    MAPE_DP=('MAPE_Sondagem', 'std'),
    MedidaB_Media=('Medida_B', 'mean'),
    MedidaB_DP=('Medida_B', 'std'),
    Amostra_Media=('amostra_obtida', 'mean'),
    TaxaResp_Media=('taxa_resposta', 'mean')
).sort_values(by='N_Sondagens', ascending=False).round(2)

print("\n--- 2. Desvios e Metadados por Instituto ---")
display(tabela_por_instituto)

# 3. Resumo por Método de Recolha
tabela_por_modo = df.groupby('modo_recolha', as_index=False).agg(
    N_Sondagens=('n_deposito', 'count'),
    Mosteller3_Media=('Mosteller_3', 'mean'),
    Mosteller3_DP=('Mosteller_3', 'std'),
    MAPE_Media=('MAPE_Sondagem', 'mean'),
    MAPE_DP=('MAPE_Sondagem', 'std'),
    MedidaB_Media=('Medida_B', 'mean'),
    MedidaB_DP=('Medida_B', 'std'),
    Amostra_Media=('amostra_obtida', 'mean'),
    TaxaResp_Media=('taxa_resposta', 'mean')
).sort_values(by='N_Sondagens', ascending=False).round(2)

print("\n--- 3. Desvios e Metadados por Método de Recolha ---")
display(tabela_por_modo)

# 4. Resumo por Instituto e Método de Recolha
tabela_instituto_modo = df.groupby(['instituto', 'modo_recolha'], as_index=False).agg(
    N_Sondagens=('n_deposito', 'count'),
    Mosteller3_Media=('Mosteller_3', 'mean'),
    Mosteller3_DP=('Mosteller_3', 'std'),
    MAPE_Media=('MAPE_Sondagem', 'mean'),
    MAPE_DP=('MAPE_Sondagem', 'std'),
    MedidaB_Media=('Medida_B', 'mean'),
    MedidaB_DP=('Medida_B', 'std'),
    Amostra_Media=('amostra_obtida', 'mean'),
    TaxaResp_Media=('taxa_resposta', 'mean')
).sort_values(by=['instituto', 'N_Sondagens'], ascending=[True, False]).round(2)

print("\n--- 4. Desvios e Metadados por Instituto e Método de Recolha ---")
display(tabela_instituto_modo)

--- 1. Desvios e Metadados por Eleição Legislativa ---


,ciclo_eleitoral,N_Sondagens,Mosteller3_Media,Mosteller3_DP,Mosteller3_Mediana,MAPE_Media,MAPE_DP,MedidaB_Media,MedidaB_DP,Amostra_Media,TaxaResp_Media
0,2011-06-05,99,3.43,1.78,2.86,22.56,10.73,0.24,0.11,894.76,57.99
1,2015-10-04,169,3.43,1.41,3.42,21.19,7.08,0.28,0.11,881.27,69.86
2,2019-10-06,119,1.88,0.82,1.98,15.69,6.38,0.20,0.09,855.97,75.19
3,2022-01-30,105,2.03,0.75,1.97,38.28,19.14,0.37,0.17,814.80,56.98
4,2024-03-10,87,2.23,1.01,1.90,29.63,10.13,0.32,0.10,764.33,57.26
5,2025-05-18,61,2.33,0.71,2.23,32.96,17.59,0.31,0.11,784.90,50.65



--- 2. Desvios e Metadados por Instituto ---


,instituto,N_Sondagens,Mosteller3_Media,Mosteller3_DP,MAPE_Media,MAPE_DP,MedidaB_Media,MedidaB_DP,Amostra_Media,TaxaResp_Media
0,Aximage,156,2.78,1.39,23.91,12.69,0.26,0.12,676.79,76.18
4,Eurosondagem,149,2.96,1.29,23.62,10.29,0.30,0.12,1028.68,76.02
6,Intercampus,99,2.72,1.03,39.06,20.99,0.36,0.15,760.86,58.44
9,Pitagórica,91,1.99,1.17,20.94,8.96,0.25,0.12,654.73,62.71
1,CESOP-U.Católica,58,2.27,1.15,20.32,9.35,0.23,0.09,1321.31,48.20
7,Marktest,34,4.11,2.11,24.92,10.80,0.30,0.13,815.41,18.82
5,ICS/ISCTE/GFK Metris,25,1.96,0.74,25.51,11.09,0.29,0.12,873.44,27.87
3,Duplimétrica/IPESPE,15,1.45,0.30,19.31,5.08,0.22,0.06,613.33,57.69
2,Consulmark2,9,1.61,0.71,23.90,7.22,0.26,0.07,679.89,59.52
8,Outro,4,2.63,1.07,24.59,14.77,0.25,0.10,1008.00,78.10



--- 3. Desvios e Metadados por Método de Recolha ---


,modo_recolha,N_Sondagens,Mosteller3_Media,Mosteller3_DP,MAPE_Media,MAPE_DP,MedidaB_Media,MedidaB_DP,Amostra_Media,TaxaResp_Media
3,Telefone,541,2.72,1.41,25.13,14.22,0.28,0.13,812.41,64.55
2,Presencial,53,2.38,1.23,20.96,10.28,0.25,0.11,1170.19,45.14
1,Internet e Telefone,45,2.14,0.61,34.68,13.32,0.32,0.11,828.18,69.87
0,Internet,1,1.05,NaN,24.51,NaN,0.20,NaN,504.00,89.84



--- 4. Desvios e Metadados por Instituto e Método de Recolha ---


,instituto,modo_recolha,N_Sondagens,Mosteller3_Media,Mosteller3_DP,MAPE_Media,MAPE_DP,MedidaB_Media,MedidaB_DP,Amostra_Media,TaxaResp_Media
2,Aximage,Telefone,110,3.06,1.52,19.49,9.48,0.24,0.12,616.44,78.64
1,Aximage,Internet e Telefone,45,2.14,0.61,34.68,13.32,0.32,0.11,828.18,69.87
0,Aximage,Internet,1,1.05,NaN,24.51,NaN,0.20,NaN,504.00,89.84
4,CESOP-U.Católica,Telefone,36,1.89,0.64,22.20,9.65,0.23,0.09,1215.42,42.27
3,CESOP-U.Católica,Presencial,22,2.90,1.50,17.24,8.14,0.21,0.10,1494.59,57.91
5,Consulmark2,Telefone,9,1.61,0.71,23.90,7.22,0.26,0.07,679.89,59.52
6,Duplimétrica/IPESPE,Telefone,15,1.45,0.30,19.31,5.08,0.22,0.06,613.33,57.69
8,Eurosondagem,Telefone,148,2.96,1.29,23.69,10.29,0.30,0.12,1021.80,75.98
7,Eurosondagem,Presencial,1,2.39,NaN,13.24,NaN,0.16,NaN,2048.00,81.10
9,ICS/ISCTE/GFK Metris,Presencial,24,1.92,0.72,25.41,11.31,0.29,0.12,876.42,28.41


## 2.4. Verificação de Valores Zero / Atípicos nas Variáveis Operacionais

In [ ]:
# Identificar sondagens com valor 0 em amostra_obtida ou variáveis de entrevistadores
filtro_zeros = (
    (df['amostra_obtida'] == 0) |
    (df['n_entrevistadores'] == 0) |
    (df['n_min_entrevistas'] == 0) |
    (df['n_max_entrevistas'] == 0)
)

colunas_inspecao = [
    'n_deposito', 'instituto', 'data_recolha', 'modo_recolha',
    'amostra_obtida', 'taxa_resposta', 'n_entrevistadores',
    'n_min_entrevistas', 'n_max_entrevistas'
]

print(f"Sondagens com valores zero em variáveis operacionais ({filtro_zeros.sum()} casos):")
display(df.loc[filtro_zeros, colunas_inspecao])

Sondagens com valores zero em variáveis operacionais (4 casos):


,n_deposito,instituto,data_recolha,modo_recolha,amostra_obtida,taxa_resposta,n_entrevistadores,n_min_entrevistas,n_max_entrevistas
100,2023049,Aximage,2023-11-13,Internet,504,89.84,0,0,0
531,2011082,Marktest,2011-10-22,Telefone,0,25.40,42,2,36
616,2010023,Intercampus,2010-04-30,Telefone,803,35.70,58,0,0
625,2010009,Intercampus,2010-02-27,Telefone,1015,29.60,47,0,0


## 2.5. Matriz de Correlações das Variáveis Quantitativas

In [ ]:
# Matriz de Correlações de Pearson das variáveis quantitativas (Pedido 2)
matriz_correlacoes = df[variaveis_quantitativas].corr(method='pearson').round(3)
matriz_correlacoes_export = matriz_correlacoes.reset_index().rename(columns={'index': 'Variável'})

print("--- Matriz de Correlações de Pearson (Variáveis Quantitativas) ---")
display(matriz_correlacoes_export)

--- Matriz de Correlações de Pearson (Variáveis Quantitativas) ---


,Variável,Mosteller_3,MAPE_Sondagem,Medida_B,amostra_obtida,taxa_resposta,n_entrevistadores,n_min_entrevistas,n_max_entrevistas,dias_ate_prox_eleicao,dias_desde_ulti_eleicao,PS,PSD-CDS,BE,CDU,PAN,L,IL,CH,OBN
0,Mosteller_3,1.000,0.394,0.574,-0.040,0.064,0.044,0.000,-0.053,0.466,-0.360,0.019,0.245,-0.033,0.377,0.106,0.084,0.324,0.020,0.185
1,MAPE_Sondagem,0.394,1.000,0.899,-0.156,-0.080,-0.052,-0.050,-0.084,0.090,-0.446,-0.075,-0.392,0.126,-0.217,0.468,-0.064,0.009,-0.130,-0.246
2,Medida_B,0.574,0.899,1.000,-0.108,-0.040,-0.070,0.092,0.049,0.244,-0.410,-0.001,-0.263,-0.008,-0.076,0.283,-0.210,-0.093,-0.227,-0.059
3,amostra_obtida,-0.040,-0.156,-0.108,1.000,-0.075,0.223,0.210,0.324,-0.083,0.147,0.019,0.110,-0.029,0.077,-0.115,0.004,-0.158,-0.071,0.124
4,taxa_resposta,0.064,-0.080,-0.040,-0.075,1.000,-0.486,0.349,0.230,0.276,0.140,0.276,0.061,0.222,0.306,0.167,-0.145,0.030,-0.178,0.079
5,n_entrevistadores,0.044,-0.052,-0.070,0.223,-0.486,1.000,-0.344,-0.306,-0.114,-0.026,-0.147,0.159,-0.045,0.027,0.004,-0.083,0.090,0.127,0.056
6,n_min_entrevistas,0.000,-0.050,0.092,0.210,0.349,-0.344,1.000,0.926,0.262,0.088,0.274,-0.011,0.067,0.248,-0.161,-0.117,-0.413,-0.320,0.242
7,n_max_entrevistas,-0.053,-0.084,0.049,0.324,0.230,-0.306,0.926,1.000,0.206,0.115,0.223,-0.026,0.004,0.178,-0.255,0.101,-0.290,-0.178,0.203
8,dias_ate_prox_eleicao,0.466,0.090,0.244,-0.083,0.276,-0.114,0.262,0.206,1.000,-0.476,0.125,0.210,0.247,0.411,-0.015,-0.291,-0.159,-0.390,0.213
9,dias_desde_ulti_eleicao,-0.360,-0.446,-0.410,0.147,0.140,-0.026,0.088,0.115,-0.476,1.000,0.247,-0.120,0.030,0.270,0.194,-0.286,-0.334,-0.281,0.453


## 2.6. Demonstração da Relação entre Instituto e Cliente

In [ ]:
from scipy.stats import chi2_contingency

df['cliente_tratado'] = df['cliente'].fillna('Não especificado')

# 1. Tabela de Concentração por Instituto (Top 1 e Top 2 Clientes)
linhas_concentracao = []
for inst, grupo in df.groupby('instituto'):
    contagem_cli = grupo['cliente_tratado'].value_counts()
    n_total = len(grupo)

    top1_nome = contagem_cli.index[0]
    top1_n    = contagem_cli.iloc[0]
    top1_pct  = (top1_n / n_total) * 100.0

    top2_nome = contagem_cli.index[1] if len(contagem_cli) > 1 else '-'
    top2_n    = contagem_cli.iloc[1] if len(contagem_cli) > 1 else 0
    top2_pct  = ((top1_n + top2_n) / n_total) * 100.0

    linhas_concentracao.append({
        'Instituto': inst,
        'N_Sondagens': n_total,
        'N_Clientes_Distintos': len(contagem_cli),
        'Cliente_Principal (Top 1)': top1_nome,
        'N_Top1': top1_n,
        '%_Cliente_Principal': round(top1_pct, 1),
        'Segundo_Cliente (Top 2)': top2_nome,
        'N_Top2': top2_n,
        '%_Acumulada_Top2': round(top2_pct, 1)
    })

tabela_concentracao_clientes = pd.DataFrame(linhas_concentracao).sort_values(by='N_Sondagens', ascending=False)

# 2. Matrizes Cruzadas Completas (Instituto x Cliente em Contagem N e em % de Linha)
crosstab_inst_cli_n = pd.crosstab(df['instituto'], df['cliente_tratado'], margins=True, margins_name='Total').reset_index()
crosstab_inst_cli_pct = (pd.crosstab(df['instituto'], df['cliente_tratado'], normalize='index') * 100.0).round(1).reset_index()

# 3. Teste Qui-Quadrado e V de Cramér (Medida formal de associação nominal entre 0 e 1)
ct_base = pd.crosstab(df['instituto'], df['cliente_tratado'])
chi2, p_val_chi2, gl, _ = chi2_contingency(ct_base)
n_obs = ct_base.sum().sum()
v_cramer = np.sqrt(chi2 / (n_obs * (min(ct_base.shape) - 1)))

print(f"Teste de Independência Instituto vs. Cliente: Qui-Quadrado = {chi2:.2f} (gl = {gl}, p = {p_val_chi2:.4e}) | V de Cramér = {v_cramer:.4f}")
print("\n--- 1. Quadro Resumo de Concentração de Clientes por Instituto ---")
display(tabela_concentracao_clientes)

print("\n--- 2. Tabela Cruzada Instituto x Cliente (Número de Sondagens) ---")
display(crosstab_inst_cli_n)

print("\n--- 3. Tabela Cruzada Instituto x Cliente (% dentro de cada Instituto) ---")
display(crosstab_inst_cli_pct)

Teste de Independência Instituto vs. Cliente: Qui-Quadrado = 2569.19 (gl = 108, p = 0.0000e+00) | V de Cramér = 0.6679

--- 1. Quadro Resumo de Concentração de Clientes por Instituto ---


,Instituto,N_Sondagens,N_Clientes_Distintos,Cliente_Principal (Top 1),N_Top1,%_Cliente_Principal,Segundo_Cliente (Top 2),N_Top2,%_Acumulada_Top2
0,Aximage,156,5,Grupo Cofina,111,71.2,Global Media,33,92.3
4,Eurosondagem,149,2,Grupo Impresa,120,80.5,Newsplex,29,100.0
6,Intercampus,99,4,Grupo Cofina,66,66.7,Grupo Media Capital,31,98.0
9,Pitagórica,91,3,Grupo Media Capital,66,72.5,Newsplex,17,91.2
1,CESOP-U.Católica,58,2,RTP,57,98.3,Não especificado,1,100.0
7,Marktest,34,2,Ongoing Media,20,58.8,Não especificado,14,100.0
5,ICS/ISCTE/GFK Metris,25,2,Grupo Impresa,23,92.0,Partido Socialista,2,100.0
3,Duplimétrica/IPESPE,15,1,Grupo Media Capital,15,100.0,-,0,100.0
2,Consulmark2,9,1,Newsplex,9,100.0,-,0,100.0
8,Outro,4,3,Não especificado,2,50.0,Grupo Media Capital,1,75.0



--- 2. Tabela Cruzada Instituto x Cliente (Número de Sondagens) ---


cliente_tratado,instituto,Global Media,Grupo Cofina,Grupo Impresa,Grupo Media Capital,Jornal Folha Nacional,Jornal Público,Newsplex,Não especificado,O Jornal Económico,Ongoing Media,Partido Socialista,RTP,Revista FOCUS,Total
0,Aximage,33,111,0,2,5,0,0,0,5,0,0,0,0,156
1,CESOP-U.Católica,0,0,0,0,0,0,0,1,0,0,0,57,0,58
2,Consulmark2,0,0,0,0,0,0,9,0,0,0,0,0,0,9
3,Duplimétrica/IPESPE,0,0,0,15,0,0,0,0,0,0,0,0,0,15
4,Eurosondagem,0,0,120,0,0,0,29,0,0,0,0,0,0,149
5,ICS/ISCTE/GFK Metris,0,0,23,0,0,0,0,0,0,0,2,0,0,25
6,Intercampus,0,66,0,31,0,1,0,0,0,0,1,0,0,99
7,Marktest,0,0,0,0,0,0,0,14,0,20,0,0,0,34
8,Outro,0,0,0,1,0,0,0,2,0,0,0,0,1,4
9,Pitagórica,8,0,0,66,0,0,17,0,0,0,0,0,0,91



--- 3. Tabela Cruzada Instituto x Cliente (% dentro de cada Instituto) ---


cliente_tratado,instituto,Global Media,Grupo Cofina,Grupo Impresa,Grupo Media Capital,Jornal Folha Nacional,Jornal Público,Newsplex,Não especificado,O Jornal Económico,Ongoing Media,Partido Socialista,RTP,Revista FOCUS
0,Aximage,21.2,71.2,0.0,1.3,3.2,0.0,0.0,0.0,3.2,0.0,0.0,0.0,0.0
1,CESOP-U.Católica,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.7,0.0,0.0,0.0,98.3,0.0
2,Consulmark2,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Duplimétrica/IPESPE,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Eurosondagem,0.0,0.0,80.5,0.0,0.0,0.0,19.5,0.0,0.0,0.0,0.0,0.0,0.0
5,ICS/ISCTE/GFK Metris,0.0,0.0,92.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,0.0,0.0
6,Intercampus,0.0,66.7,0.0,31.3,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
7,Marktest,0.0,0.0,0.0,0.0,0.0,0.0,0.0,41.2,0.0,58.8,0.0,0.0,0.0
8,Outro,0.0,0.0,0.0,25.0,0.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,25.0
9,Pitagórica,8.8,0.0,0.0,72.5,0.0,0.0,18.7,0.0,0.0,0.0,0.0,0.0,0.0


## 2.7. Exportar todas as tabelas para um ficheiro Excel limpo no Google Drive

In [ ]:
caminho_saida = '/content/drive/MyDrive/A Máquina de Sondagens/Estatisticas_Descritivas_Cap3.xlsx'

with pd.ExcelWriter(caminho_saida) as writer:
    tabela_quant.to_excel(writer, sheet_name='1_Quantitativas', index=False)
    tabela_cat.to_excel(writer, sheet_name='2_Categoricas', index=False)
    tabela_por_eleicao.to_excel(writer, sheet_name='3_Por_Eleicao', index=False)
    tabela_por_instituto.to_excel(writer, sheet_name='4_Por_Instituto', index=False)
    tabela_por_modo.to_excel(writer, sheet_name='5_Por_Modo', index=False)
    tabela_instituto_modo.to_excel(writer, sheet_name='6_Instituto_x_Modo', index=False)
    matriz_correlacoes_export.to_excel(writer, sheet_name='7_Matriz_Correlacoes', index=False)
    tabela_concentracao_clientes.to_excel(writer, sheet_name='8_Concentracao_Clientes', index=False)
    crosstab_inst_cli_n.to_excel(writer, sheet_name='9_Cruzada_Inst_Cli_N', index=False)
    crosstab_inst_cli_pct.to_excel(writer, sheet_name='10_Cruzada_Inst_Cli_Pct', index=False)
    df[['n_deposito', 'instituto', 'data_recolha', 'Mosteller_3', 'MAPE_Sondagem', 'Medida_B']].to_excel(
        writer, sheet_name='11_Variaveis_Alvo', index=False
    )

print("Ficheiro Excel atualizado exportado com sucesso para:", caminho_saida)

Ficheiro Excel atualizado exportado com sucesso para: /content/drive/MyDrive/A Máquina de Sondagens/Estatisticas_Descritivas_Cap3.xlsx


# 3\. Aplicação dos modelos

## 3.1. Modelação Preditiva: Preparação de Variáveis e Interações Moderadoras

In [ ]:
# ==============================================================================
# 3.1. PREPARAÇÃO DE VARIÁVEIS, INTERAÇÕES MODERADORAS E SPLIT 70/30
# ==============================================================================
from sklearn.model_selection import RandomizedSearchCV
from sklearn.tree import export_text

LISTA_DVS = ['Mosteller_3', 'MAPE_Sondagem', 'Medida_B']

# Fundir a única sondagem 100% Internet com 'Internet e Telefone' apenas para os modelos
df['modo_modelo'] = df['modo_recolha'].replace({'Internet': 'Internet e Telefone'})

vars_quantitativas = [
    'amostra_obtida', 'taxa_resposta', 'n_entrevistadores',
    'n_min_entrevistas', 'n_max_entrevistas',
    'dias_ate_prox_eleicao', 'dias_desde_ulti_eleicao',
    'PS', 'PSD-CDS', 'BE', 'CDU', 'PAN', 'L', 'IL', 'CH', 'OBN'
]
vars_binarias = ['coligacao_PSD-CDS_na_sondagem', 'demissao_gov', 'comunic_presi']
vars_categoricas = ['instituto', 'modo_modelo']

df_features = df[vars_quantitativas + vars_binarias + vars_categoricas].copy()

for p in ['PAN', 'L', 'IL', 'CH']:
    df_features[p] = df_features[p].fillna(0.0)

# 1. Matriz Base (30 colunas - utilizada nos Modelos de Árvores 3, 4 e 5)
X_base = pd.get_dummies(df_features, columns=vars_categoricas, drop_first=True, dtype=float)

# Mapear cada coluna de X_base à sua variável de origem (soma dummies de instituto e modo_modelo)
mapa_grupos_base = {}
for col in X_base.columns:
    if col.startswith('instituto_'):
        mapa_grupos_base[col] = 'instituto'
    elif col.startswith('modo_modelo_'):
        mapa_grupos_base[col] = 'modo_modelo'
    else:
        mapa_grupos_base[col] = col

# 2. Matriz com Interações Moderadoras (206 colunas - exclusiva para os Modelos Lineares 1 e 2)
inst_cols = [c for c in X_base.columns if c.startswith('instituto_')]
modo_cols = [c for c in X_base.columns if c.startswith('modo_modelo_')]

X_inter = X_base.copy()
for d_col in inst_cols + modo_cols:
    for q_col in vars_quantitativas:
        X_inter[f'{d_col}_x_{q_col}'] = X_base[d_col] * X_base[q_col]

# Mapear cada coluna de X_inter (distinguindo efeitos principais e blocos de interação moderadora)
mapa_grupos_inter = {}
for col in X_inter.columns:
    if '_x_' in col:
        prefixo = col.split('_x_')[0]
        if prefixo.startswith('instituto_'):
            mapa_grupos_inter[col] = 'Interacao_Instituto_x_Quant'
        else:
            mapa_grupos_inter[col] = 'Interacao_Modo_x_Quant'
    else:
        mapa_grupos_inter[col] = mapa_grupos_base[col]

# 3. Divisão Treino (70% = 448 sondagens) / Teste (30% = 192 sondagens)
idx_train, idx_test = train_test_split(df.index, test_size=0.30, random_state=42)

X_train_base = X_base.loc[idx_train].copy()
X_test_base  = X_base.loc[idx_test].copy()

X_train_inter = X_inter.loc[idx_train].copy()
X_test_inter  = X_inter.loc[idx_test].copy()

scaler = StandardScaler()
X_tr_inter_sc = pd.DataFrame(scaler.fit_transform(X_train_inter), columns=X_inter.columns, index=idx_train)
X_te_inter_sc = pd.DataFrame(scaler.transform(X_test_inter), columns=X_inter.columns, index=idx_test)

y_train_dict = {dv: df.loc[idx_train, dv].copy() for dv in LISTA_DVS}
y_test_dict  = {dv: df.loc[idx_test, dv].copy() for dv in LISTA_DVS}

# 4. Dicionários globais de armazenamento
resultados_metricas = {dv: {} for dv in LISTA_DVS}
previsoes_teste     = {dv: {'Valor_Real': y_test_dict[dv]} for dv in LISTA_DVS}
tabelas_ols_dict    = {}
tabelas_enet_dict   = {}
regras_modelos_dict = {dv: [] for dv in LISTA_DVS}

importancia_detalhada_inter = {
    dv: pd.DataFrame({
        'Variavel_Detalhada': X_inter.columns,
        'Variavel_Origem': [mapa_grupos_inter[c] for c in X_inter.columns]
    }) for dv in LISTA_DVS
}

importancia_detalhada_base = {
    dv: pd.DataFrame({
        'Variavel_Detalhada': X_base.columns,
        'Variavel_Origem': [mapa_grupos_base[c] for c in X_base.columns]
    }) for dv in LISTA_DVS
}

print(f"Matriz Base (Modelos 3, 4 e 5): {X_train_base.shape[1]} colunas")
print(f"Matriz com Interações Moderadoras (Modelos 1 e 2): {X_train_inter.shape[1]} colunas")
print(f"Treino (70%): {len(idx_train)} sondagens (5% min_samples = {int(np.ceil(len(idx_train)*0.05))} sondagens) | Teste (30%): {len(idx_test)} sondagens")

Matriz Base (Modelos 3, 4 e 5): 30 colunas
Matriz com Interações Moderadoras (Modelos 1 e 2): 206 colunas
Treino (70%): 448 sondagens (5% min_samples = 23 sondagens) | Teste (30%): 192 sondagens


## 3.2. Registo de Recursos Computacionais (CPU, RAM e Tempos de Execução)

In [ ]:
import psutil
import platform

info_so  = f"{platform.system()} {platform.release()}"
info_cpu = f"{platform.processor()} ({psutil.cpu_count(logical=False)} núcleos físicos / {psutil.cpu_count(logical=True)} lógicos)"
info_ram = f"{psutil.virtual_memory().total / (1024**3):.2f} GB"

print("=== DIAGNÓSTICO DE RECURSOS COMPUTACIONAIS (GOOGLE COLAB) ===")
print(f"Sistema Operativo: {info_so}")
print(f"Processador (CPU): {info_cpu}")
print(f"Memória RAM Total: {info_ram} | Disponível: {psutil.virtual_memory().available / (1024**3):.2f} GB")

registo_computacional = []

def registar_custo(nome_modelo, dv, tempo_seg, notas=""):
    global registo_computacional
    # Evita duplicados se a célula for corrida mais do que uma vez
    registo_computacional = [r for r in registo_computacional if not (r['Modelo'] == nome_modelo and r['Variavel_Dependente'] == dv)]
    registo_computacional.append({
        'Variavel_Dependente': dv,
        'Modelo': nome_modelo,
        'Tempo_Execucao (s)': round(tempo_seg, 4),
        'Hardware_CPU': info_cpu,
        'RAM_Total': info_ram,
        'Detalhes_Processamento': notas
    })

=== DIAGNÓSTICO DE RECURSOS COMPUTACIONAIS (GOOGLE COLAB) ===
Sistema Operativo: Linux 6.6.159+
Processador (CPU): x86_64 (1 núcleos físicos / 2 lógicos)
Memória RAM Total: 12.67 GB | Disponível: 10.91 GB


## 3.3. Modelo 1: Regressão Linear com Seleção Iterativa Stepwise (Data Mining Lineares)

In [ ]:
# ==============================================================================
# 3.3. MODELO 1: OLS COM SELEÇÃO STEPWISE ITERATIVA E INTERAÇÕES MODERADORAS
# ==============================================================================
resumo_ols_step = []

def spss_stepwise_ols(X_train_df, y_train_s, p_enter=0.05, p_remove=0.10):
    included = []
    step = 0
    while True:
        changed = False
        excluded = list(set(X_train_df.columns) - set(included))
        best_pval = 1.0
        best_feature = None
        for new_col in excluded:
            try:
                model = sm.OLS(y_train_s, sm.add_constant(X_train_df[included + [new_col]])).fit()
                pval = model.pvalues[new_col]
                if pval < best_pval:
                    best_pval = pval
                    best_feature = new_col
            except Exception:
                continue
        if best_feature is not None and best_pval < p_enter:
            included.append(best_feature)
            changed = True
            step += 1
        if included:
            model = sm.OLS(y_train_s, sm.add_constant(X_train_df[included])).fit()
            pvalues = model.pvalues.drop('const')
            max_pval = pvalues.max()
            if max_pval > p_remove:
                worst_feature = pvalues.idxmax()
                included.remove(worst_feature)
                changed = True
                step += 1
        if not changed:
            break
    return included, step

for dv in LISTA_DVS:
    y_tr, y_te = y_train_dict[dv], y_test_dict[dv]

    t0 = time.time()
    selected_vars, n_iter = spss_stepwise_ols(X_tr_inter_sc, y_tr, p_enter=0.05, p_remove=0.10)

    X_tr_sel = sm.add_constant(X_tr_inter_sc[selected_vars])
    X_te_sel = sm.add_constant(X_te_inter_sc[selected_vars], has_constant='add')

    ols_final = sm.OLS(y_tr, X_tr_sel).fit(cov_type='HC3')
    y_pred_tr_ols = ols_final.predict(X_tr_sel)
    y_pred_te_ols = ols_final.predict(X_te_sel)
    tempo_ols = time.time() - t0

    params_ols = f"Stepwise ({n_iter} iterações, {len(selected_vars)}/206 vars selecionadas)"
    registar_custo('1_OLS_Stepwise', dv, tempo_ols, params_ols)

    # Guardar importância percentual na tabela de 206 colunas (0 para não selecionadas)
    coefs_series = pd.Series(0.0, index=X_inter.columns)
    coefs_series.loc[selected_vars] = ols_final.params.drop('const').values
    abs_coefs = coefs_series.abs()
    imp_pct_series = (abs_coefs / abs_coefs.sum()) * 100.0 if abs_coefs.sum() > 0 else 0.0

    importancia_detalhada_inter[dv]['OLS_Coef_Beta'] = coefs_series.values
    importancia_detalhada_inter[dv]['OLS_Imp_%']     = imp_pct_series.values

    # Tabela detalhada das variáveis selecionadas pelo Stepwise OLS
    df_ols_sel = pd.DataFrame({
        'Variavel_Detalhada': selected_vars,
        'Variavel_Origem': [mapa_grupos_inter[c] for c in selected_vars],
        'Coef_Padronizado_Beta': ols_final.params.drop('const').values,
        'Erro_Padrao_Robusto': ols_final.bse.drop('const').values,
        'Estatistica_t': ols_final.tvalues.drop('const').values,
        'p_value': ols_final.pvalues.drop('const').values,
        'OLS_Imp_%': imp_pct_series.loc[selected_vars].values
    }).sort_values(by='OLS_Imp_%', ascending=False).round(4)
    tabelas_ols_dict[dv] = df_ols_sel

    mae_tr  = mean_absolute_error(y_tr, y_pred_tr_ols)
    rmse_tr = np.sqrt(mean_squared_error(y_tr, y_pred_tr_ols))
    r2_tr   = r2_score(y_tr, y_pred_tr_ols)

    mae_te  = mean_absolute_error(y_te, y_pred_te_ols)
    rmse_te = np.sqrt(mean_squared_error(y_te, y_pred_te_ols))
    mape_te = np.mean(np.abs((y_te - y_pred_te_ols) / y_te)) * 100.0
    r2_te   = r2_score(y_te, y_pred_te_ols)

    resultados_metricas[dv]['1_OLS_Stepwise'] = {
        'Variavel_Dependente': dv, 'Modelo': '1_OLS_Stepwise',
        'MAE_Treino': round(mae_tr, 4), 'RMSE_Treino': round(rmse_tr, 4), 'R2_Treino': round(r2_tr, 4),
        'MAE_Teste': round(mae_te, 4), 'RMSE_Teste': round(rmse_te, 4),
        'MAPE_Teste (%)': round(mape_te, 2), 'R2_Teste': round(r2_te, 4),
        'Tempo_Treino (s)': round(tempo_ols, 4),
        'Hiperparametros': params_ols
    }
    previsoes_teste[dv]['1_OLS_Stepwise'] = y_pred_te_ols
    resumo_ols_step.append(resultados_metricas[dv]['1_OLS_Stepwise'])

print("=== DESEMPENHO DO MODELO 1 (OLS STEPWISE COM INTERAÇÕES) ===")
display(pd.DataFrame(resumo_ols_step))

/tmp/ipykernel_1303/1946332869.py:16: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model = sm.OLS(y_train_s, sm.add_constant(X_train_df[included + [new_col]])).fit()
/tmp/ipykernel_1303/1946332869.py:16: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model = sm.OLS(y_train_s, sm.add_constant(X_train_df[included + [new_col]])).fit()
/tmp/ipykernel_1303/1946332869.py:16: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model = sm.OLS(y_train_s, sm.add_constant(X_train_df[included + [new_col]])).fit()
/tmp/ipykernel_1303/1946332869.py:16: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model = sm.OLS(y_train_s, sm.add_constant(X_train_df[included + [new_col]])).fit()
/tmp/ipykernel_1303/1946332869.py:16: SingularMatrixWarning: The des

=== DESEMPENHO DO MODELO 1 (OLS STEPWISE COM INTERAÇÕES) ===


,Variavel_Dependente,Modelo,MAE_Treino,RMSE_Treino,R2_Treino,MAE_Teste,RMSE_Teste,MAPE_Teste (%),R2_Teste,Tempo_Treino (s),Hiperparametros
0,Mosteller_3,1_OLS_Stepwise,0.6658,0.9008,0.5826,0.6584,0.897,29.01,0.5226,12.2750,"Stepwise (21 iterações, 17/206 vars selecionadas)"
1,MAPE_Sondagem,1_OLS_Stepwise,5.3738,6.9228,0.7481,5.8368,7.833,28.44,0.7203,15.5931,"Stepwise (23 iterações, 21/206 vars selecionadas)"
2,Medida_B,1_OLS_Stepwise,0.0654,0.0842,0.5783,0.0677,0.084,29.52,0.5676,17.9306,"Stepwise (29 iterações, 23/206 vars selecionadas)"


## 3.4. Modelo 2: Regressão Regularizada com Interações (Elastic Net)

In [ ]:
# ==============================================================================
# 3.4. MODELO 2: ELASTIC NET COM INTERAÇÕES MODERADORAS
# ==============================================================================
resumo_enet = []

for dv in LISTA_DVS:
    y_tr, y_te = y_train_dict[dv], y_test_dict[dv]

    t1 = time.time()
    modelo_enet = ElasticNetCV(
        l1_ratio=[0.2, 0.5, 0.8, 1.0],
        alphas=np.logspace(-4, 1, 50),
        cv=5,
        random_state=42,
        max_iter=10000
    )
    modelo_enet.fit(X_tr_inter_sc, y_tr)

    y_pred_tr_en = modelo_enet.predict(X_tr_inter_sc)
    y_pred_te_en = modelo_enet.predict(X_te_inter_sc)
    tempo_enet = time.time() - t1

    n_ativos = int(np.sum(modelo_enet.coef_ != 0))
    params_enet = f"alpha={modelo_enet.alpha_:.4f}, l1_ratio={modelo_enet.l1_ratio_} ({n_ativos}/206 vars ativas)"
    registar_custo('2_ElasticNet', dv, tempo_enet, params_enet)

    abs_coef = np.abs(modelo_enet.coef_)
    imp_enet_pct = (abs_coef / abs_coef.sum()) * 100.0 if abs_coef.sum() > 0 else np.zeros_like(abs_coef)

    importancia_detalhada_inter[dv]['ElasticNet_Coef']  = modelo_enet.coef_
    importancia_detalhada_inter[dv]['ElasticNet_Imp_%'] = imp_enet_pct

    # Guardar tabela das variáveis ativas (coef != 0) selecionadas pelo Elastic Net
    mask_ativo = modelo_enet.coef_ != 0
    tabelas_enet_dict[dv] = pd.DataFrame({
        'Variavel_Detalhada': X_inter.columns[mask_ativo],
        'Variavel_Origem': [mapa_grupos_inter[c] for c in X_inter.columns[mask_ativo]],
        'ElasticNet_Coef_Padronizado': modelo_enet.coef_[mask_ativo],
        'ElasticNet_Imp_%': imp_enet_pct[mask_ativo]
    }).sort_values(by='ElasticNet_Imp_%', ascending=False).round(4)

    mae_tr_en  = mean_absolute_error(y_tr, y_pred_tr_en)
    rmse_tr_en = np.sqrt(mean_squared_error(y_tr, y_pred_tr_en))
    r2_tr_en   = r2_score(y_tr, y_pred_tr_en)

    mae_te_en  = mean_absolute_error(y_te, y_pred_te_en)
    rmse_te_en = np.sqrt(mean_squared_error(y_te, y_pred_te_en))
    mape_te_en = np.mean(np.abs((y_te - y_pred_te_en) / y_te)) * 100.0
    r2_te_en   = r2_score(y_te, y_pred_te_en)

    resultados_metricas[dv]['2_ElasticNet'] = {
        'Variavel_Dependente': dv, 'Modelo': '2_ElasticNet',
        'MAE_Treino': round(mae_tr_en, 4), 'RMSE_Treino': round(rmse_tr_en, 4), 'R2_Treino': round(r2_tr_en, 4),
        'MAE_Teste': round(mae_te_en, 4), 'RMSE_Teste': round(rmse_te_en, 4),
        'MAPE_Teste (%)': round(mape_te_en, 2), 'R2_Teste': round(r2_te_en, 4),
        'Tempo_Treino (s)': round(tempo_enet, 4), 'Hiperparametros': params_enet
    }
    previsoes_teste[dv]['2_ElasticNet'] = y_pred_te_en
    resumo_enet.append(resultados_metricas[dv]['2_ElasticNet'])

print("=== DESEMPENHO DO MODELO 2 (ELASTIC NET COM INTERAÇÕES) ===")
display(pd.DataFrame(resumo_enet))

=== DESEMPENHO DO MODELO 2 (ELASTIC NET COM INTERAÇÕES) ===


,Variavel_Dependente,Modelo,MAE_Treino,RMSE_Treino,R2_Treino,MAE_Teste,RMSE_Teste,MAPE_Teste (%),R2_Teste,Tempo_Treino (s),Hiperparametros
0,Mosteller_3,2_ElasticNet,0.6264,0.8590,0.6205,0.6261,0.8988,26.97,0.5206,38.9101,"alpha=0.0569, l1_ratio=0.2 (71/206 vars ativas)"
1,MAPE_Sondagem,2_ElasticNet,4.8167,6.1786,0.7994,5.7990,7.5461,27.94,0.7404,57.6757,"alpha=0.0720, l1_ratio=1.0 (69/206 vars ativas)"
2,Medida_B,2_ElasticNet,0.0744,0.0943,0.4712,0.0713,0.0894,32.65,0.5102,12.1585,"alpha=0.0043, l1_ratio=1.0 (22/206 vars ativas)"


## 3.5. Modelo 3: Árvore de Decisão CART (Com Restrição de 5% de Amostra Mínima por Folha)

In [ ]:
# ==============================================================================
# 3.5. MODELO 3: ÁRVORE DE DECISÃO CART (COM RESTRIÇÃO DE 5% POR FOLHA)
# ==============================================================================
resumo_cart = []

# Profundidade livre/alargada, controlada pelo mínimo de 5% da amostra de treino por folha
grelha_cart = {
    'max_depth': [None, 10, 15, 20],
    'min_samples_split': [0.05, 0.10],
    'min_samples_leaf': [0.05]
}

for dv in LISTA_DVS:
    t0 = time.time()
    y_tr, y_te = y_train_dict[dv], y_test_dict[dv]

    grid_cart = GridSearchCV(
        DecisionTreeRegressor(random_state=42),
        param_grid=grelha_cart,
        scoring='neg_mean_absolute_error',
        cv=5,
        n_jobs=-1
    )
    grid_cart.fit(X_train_base, y_tr)
    modelo_cart = grid_cart.best_estimator_

    y_pred_tr = modelo_cart.predict(X_train_base)
    y_pred_te = modelo_cart.predict(X_test_base)
    tempo_cart = time.time() - t0

    params_cart = f"{grid_cart.best_params_} (profundidade efetiva={modelo_cart.get_depth()}, folhas={modelo_cart.get_n_leaves()})"
    registar_custo('3_DecisionTree_CART', dv, tempo_cart, params_cart)

    # Extrair e guardar as regras completas da Árvore CART
    texto_regras_cart = export_text(modelo_cart, feature_names=list(X_train_base.columns), max_depth=20)
    regras_modelos_dict[dv] = [r for r in regras_modelos_dict[dv] if r['Modelo'] != '3_DecisionTree_CART (Árvore Completa)']
    regras_modelos_dict[dv].append({
        'Variavel_Dependente': dv,
        'Modelo': '3_DecisionTree_CART (Árvore Completa)',
        'Profundidade_Efetiva': modelo_cart.get_depth(),
        'N_Folhas': modelo_cart.get_n_leaves(),
        'Regras_de_Decisao': texto_regras_cart
    })

    perm_cart = permutation_importance(modelo_cart, X_test_base, y_te, n_repeats=15, random_state=42, scoring='neg_mean_absolute_error')

    importancia_detalhada_base[dv]['CART_Imp_%']    = modelo_cart.feature_importances_ * 100.0
    importancia_detalhada_base[dv]['CART_Perm_MAE'] = np.maximum(0, perm_cart.importances_mean)

    mae_tr  = mean_absolute_error(y_tr, y_pred_tr)
    rmse_tr = np.sqrt(mean_squared_error(y_tr, y_pred_tr))
    r2_tr   = r2_score(y_tr, y_pred_tr)

    mae_te  = mean_absolute_error(y_te, y_pred_te)
    rmse_te = np.sqrt(mean_squared_error(y_te, y_pred_te))
    mape_te = np.mean(np.abs((y_te - y_pred_te) / y_te)) * 100.0
    r2_te   = r2_score(y_te, y_pred_te)

    resultados_metricas[dv]['3_DecisionTree_CART'] = {
        'Variavel_Dependente': dv, 'Modelo': '3_DecisionTree_CART',
        'MAE_Treino': round(mae_tr, 4), 'RMSE_Treino': round(rmse_tr, 4), 'R2_Treino': round(r2_tr, 4),
        'MAE_Teste': round(mae_te, 4), 'RMSE_Teste': round(rmse_te, 4),
        'MAPE_Teste (%)': round(mape_te, 2), 'R2_Teste': round(r2_te, 4),
        'Tempo_Treino (s)': round(tempo_cart, 4), 'Hiperparametros': params_cart
    }
    previsoes_teste[dv]['3_DecisionTree_CART'] = y_pred_te
    resumo_cart.append(resultados_metricas[dv]['3_DecisionTree_CART'])

print("=== DESEMPENHO DO MODELO 3 (CART COM MIN_LEAF = 5%) ===")
display(pd.DataFrame(resumo_cart))

=== DESEMPENHO DO MODELO 3 (CART COM MIN_LEAF = 5%) ===


,Variavel_Dependente,Modelo,MAE_Treino,RMSE_Treino,R2_Treino,MAE_Teste,RMSE_Teste,MAPE_Teste (%),R2_Teste,Tempo_Treino (s),Hiperparametros
0,Mosteller_3,3_DecisionTree_CART,0.5164,0.7084,0.7419,0.5826,0.8205,26.33,0.6005,1.7992,"{'max_depth': None, 'min_samples_leaf': 0.05, ..."
1,MAPE_Sondagem,3_DecisionTree_CART,5.7946,8.1689,0.6493,6.3963,8.8978,29.26,0.6391,0.3315,"{'max_depth': None, 'min_samples_leaf': 0.05, ..."
2,Medida_B,3_DecisionTree_CART,0.0575,0.0764,0.6526,0.0647,0.0841,26.36,0.5664,0.3340,"{'max_depth': None, 'min_samples_leaf': 0.05, ..."


## 3.6. Modelo 4: Random Forest + SHAP (Com Restrição de 5% por Folha)

In [ ]:
# ==============================================================================
# 3.6. MODELO 4: RANDOM FOREST + SHAP (COM RESTRIÇÃO DE 5% POR FOLHA)
# ==============================================================================
resumo_rf = []

grelha_rf = {
    'n_estimators': [100, 150, 200],
    'max_depth': [None, 10, 15, 20],
    'min_samples_leaf': [0.05],
    'max_features': ['sqrt', 0.5, 1.0]
}

for dv in LISTA_DVS:
    t0 = time.time()
    y_tr, y_te = y_train_dict[dv], y_test_dict[dv]

    search_rf = RandomizedSearchCV(
        RandomForestRegressor(random_state=42, n_jobs=1),
        param_distributions=grelha_rf,
        n_iter=8,
        scoring='neg_mean_absolute_error',
        cv=5,
        random_state=42,
        n_jobs=-1
    )
    search_rf.fit(X_train_base, y_tr)
    modelo_rf = search_rf.best_estimator_

    y_pred_tr = modelo_rf.predict(X_train_base)
    y_pred_te = modelo_rf.predict(X_test_base)
    tempo_rf = time.time() - t0

    params_rf = str(search_rf.best_params_)
    registar_custo('4_RandomForest', dv, tempo_rf, params_rf)

    # Extrair a Árvore n.º 1 da Floresta como exemplo ilustrativo das regras internas
    arvore_exemplo_rf = modelo_rf.estimators_[0]
    texto_regras_rf = export_text(arvore_exemplo_rf, feature_names=list(X_train_base.columns), max_depth=20)
    regras_modelos_dict[dv] = [r for r in regras_modelos_dict[dv] if r['Modelo'] != '4_RandomForest (Exemplo Árvore #1 do Ensemble)']
    regras_modelos_dict[dv].append({
        'Variavel_Dependente': dv,
        'Modelo': '4_RandomForest (Exemplo Árvore #1 do Ensemble)',
        'Profundidade_Efetiva': arvore_exemplo_rf.get_depth(),
        'N_Folhas': arvore_exemplo_rf.get_n_leaves(),
        'Regras_de_Decisao': texto_regras_rf
    })

    perm_rf = permutation_importance(modelo_rf, X_test_base, y_te, n_repeats=10, random_state=42, scoring='neg_mean_absolute_error', n_jobs=1)

    explainer_rf = shap.TreeExplainer(modelo_rf)
    shap_values_rf = explainer_rf.shap_values(X_test_base, check_additivity=False)
    mean_abs_shap_rf = np.abs(shap_values_rf).mean(axis=0)

    importancia_detalhada_base[dv]['RF_MDI_Imp_%']  = modelo_rf.feature_importances_ * 100.0
    importancia_detalhada_base[dv]['RF_Perm_MAE']   = np.maximum(0, perm_rf.importances_mean)
    importancia_detalhada_base[dv]['RF_SHAP_Abs']   = mean_abs_shap_rf
    importancia_detalhada_base[dv]['RF_SHAP_Imp_%'] = (mean_abs_shap_rf / mean_abs_shap_rf.sum()) * 100.0

    mae_tr  = mean_absolute_error(y_tr, y_pred_tr)
    rmse_tr = np.sqrt(mean_squared_error(y_tr, y_pred_tr))
    r2_tr   = r2_score(y_tr, y_pred_tr)

    mae_te  = mean_absolute_error(y_te, y_pred_te)
    rmse_te = np.sqrt(mean_squared_error(y_te, y_pred_te))
    mape_te = np.mean(np.abs((y_te - y_pred_te) / y_te)) * 100.0
    r2_te   = r2_score(y_te, y_pred_te)

    resultados_metricas[dv]['4_RandomForest'] = {
        'Variavel_Dependente': dv, 'Modelo': '4_RandomForest',
        'MAE_Treino': round(mae_tr, 4), 'RMSE_Treino': round(rmse_tr, 4), 'R2_Treino': round(r2_tr, 4),
        'MAE_Teste': round(mae_te, 4), 'RMSE_Teste': round(rmse_te, 4),
        'MAPE_Teste (%)': round(mape_te, 2), 'R2_Teste': round(r2_te, 4),
        'Tempo_Treino (s)': round(tempo_rf, 4), 'Hiperparametros': params_rf
    }
    previsoes_teste[dv]['4_RandomForest'] = y_pred_te
    resumo_rf.append(resultados_metricas[dv]['4_RandomForest'])

print("=== DESEMPENHO DO MODELO 4 (RANDOM FOREST COM MIN_LEAF = 5%) ===")
display(pd.DataFrame(resumo_rf))

=== DESEMPENHO DO MODELO 4 (RANDOM FOREST COM MIN_LEAF = 5%) ===


,Variavel_Dependente,Modelo,MAE_Treino,RMSE_Treino,R2_Treino,MAE_Teste,RMSE_Teste,MAPE_Teste (%),R2_Teste,Tempo_Treino (s),Hiperparametros
0,Mosteller_3,4_RandomForest,0.5890,0.7899,0.6790,0.6078,0.8206,27.59,0.6004,8.2208,"{'n_estimators': 100, 'min_samples_leaf': 0.05..."
1,MAPE_Sondagem,4_RandomForest,5.5853,7.8750,0.6741,6.8091,9.2948,33.37,0.6062,8.2109,"{'n_estimators': 150, 'min_samples_leaf': 0.05..."
2,Medida_B,4_RandomForest,0.0602,0.0806,0.6130,0.0675,0.0850,30.38,0.5567,8.4411,"{'n_estimators': 200, 'min_samples_leaf': 0.05..."


## 3.7. Modelo 5: XGBoost + SHAP (Com min_child_weight equivalente a 5%)

In [ ]:
# ==============================================================================
# 3.7. MODELO 5: XGBOOST + SHAP (COM MIN_CHILD_WEIGHT EQUIVALENTE A 5%)
# ==============================================================================
resumo_xgb = []
min_child_samples_5pct = int(np.ceil(len(idx_train) * 0.05)) # 23 sondagens (5% do treino)

grelha_xgb = {
    'n_estimators': [100, 150],
    'learning_rate': [0.05, 0.1],
    'max_depth': [4, 6, 8, 10],
    'min_child_weight': [min_child_samples_5pct],
    'subsample': [0.8, 1.0]
}

for dv in LISTA_DVS:
    t0 = time.time()
    y_tr, y_te = y_train_dict[dv], y_test_dict[dv]

    search_xgb = RandomizedSearchCV(
        xgb.XGBRegressor(random_state=42, n_jobs=1, objective='reg:squarederror'),
        param_distributions=grelha_xgb,
        n_iter=8,
        scoring='neg_mean_absolute_error',
        cv=5,
        random_state=42,
        n_jobs=-1
    )
    search_xgb.fit(X_train_base, y_tr)
    modelo_xgb = search_xgb.best_estimator_

    y_pred_tr = modelo_xgb.predict(X_train_base)
    y_pred_te = modelo_xgb.predict(X_test_base)
    tempo_xgb = time.time() - t0

    params_xgb = str(search_xgb.best_params_)
    registar_custo('5_XGBoost', dv, tempo_xgb, params_xgb)

    # Extrair a Árvore n.º 1 do XGBoost como exemplo ilustrativo das regras de boosting
    dump_xgb = modelo_xgb.get_booster().get_dump(with_stats=True)[0]
    regras_modelos_dict[dv] = [r for r in regras_modelos_dict[dv] if r['Modelo'] != '5_XGBoost (Exemplo Árvore #1 do Boosting)']
    regras_modelos_dict[dv].append({
        'Variavel_Dependente': dv,
        'Modelo': '5_XGBoost (Exemplo Árvore #1 do Boosting)',
        'Profundidade_Efetiva': search_xgb.best_params_['max_depth'],
        'N_Folhas': dump_xgb.count('leaf='),
        'Regras_de_Decisao': dump_xgb
    })

    perm_xgb = permutation_importance(modelo_xgb, X_test_base, y_te, n_repeats=10, random_state=42, scoring='neg_mean_absolute_error', n_jobs=1)

    explainer_xgb = shap.TreeExplainer(modelo_xgb)
    shap_values_xgb = explainer_xgb.shap_values(X_test_base, check_additivity=False)
    mean_abs_shap_xgb = np.abs(shap_values_xgb).mean(axis=0)

    importancia_detalhada_base[dv]['XGB_Gain_Imp_%'] = modelo_xgb.feature_importances_ * 100.0
    importancia_detalhada_base[dv]['XGB_Perm_MAE']   = np.maximum(0, perm_xgb.importances_mean)
    importancia_detalhada_base[dv]['XGB_SHAP_Abs']   = mean_abs_shap_xgb
    importancia_detalhada_base[dv]['XGB_SHAP_Imp_%'] = (mean_abs_shap_xgb / mean_abs_shap_xgb.sum()) * 100.0

    mae_tr  = mean_absolute_error(y_tr, y_pred_tr)
    rmse_tr = np.sqrt(mean_squared_error(y_tr, y_pred_tr))
    r2_tr   = r2_score(y_tr, y_pred_tr)

    mae_te  = mean_absolute_error(y_te, y_pred_te)
    rmse_te = np.sqrt(mean_squared_error(y_te, y_pred_te))
    mape_te = np.mean(np.abs((y_te - y_pred_te) / y_te)) * 100.0
    r2_te   = r2_score(y_te, y_pred_te)

    resultados_metricas[dv]['5_XGBoost'] = {
        'Variavel_Dependente': dv, 'Modelo': '5_XGBoost',
        'MAE_Treino': round(mae_tr, 4), 'RMSE_Treino': round(rmse_tr, 4), 'R2_Treino': round(r2_tr, 4),
        'MAE_Teste': round(mae_te, 4), 'RMSE_Teste': round(rmse_te, 4),
        'MAPE_Teste (%)': round(mape_te, 2), 'R2_Teste': round(r2_te, 4),
        'Tempo_Treino (s)': round(tempo_xgb, 4), 'Hiperparametros': params_xgb
    }
    previsoes_teste[dv]['5_XGBoost'] = y_pred_te
    resumo_xgb.append(resultados_metricas[dv]['5_XGBoost'])

print("=== DESEMPENHO DO MODELO 5 (XGBOOST COM MIN_CHILD_WEIGHT = 5%) ===")
display(pd.DataFrame(resumo_xgb))

=== DESEMPENHO DO MODELO 5 (XGBOOST COM MIN_CHILD_WEIGHT = 5%) ===


,Variavel_Dependente,Modelo,MAE_Treino,RMSE_Treino,R2_Treino,MAE_Teste,RMSE_Teste,MAPE_Teste (%),R2_Teste,Tempo_Treino (s),Hiperparametros
0,Mosteller_3,5_XGBoost,0.1495,0.2276,0.9734,0.3499,0.5282,15.14,0.8344,2.5227,"{'subsample': 1.0, 'n_estimators': 100, 'min_c..."
1,MAPE_Sondagem,5_XGBoost,1.3244,2.0808,0.9772,3.6901,5.4032,16.83,0.8669,2.4478,"{'subsample': 0.8, 'n_estimators': 150, 'min_c..."
2,Medida_B,5_XGBoost,0.0147,0.0217,0.9720,0.0382,0.0549,15.52,0.8151,2.7847,"{'subsample': 0.8, 'n_estimators': 150, 'min_c..."


## 3.8. Compilação Global, Diebold-Mariano e Diagnóstico Computacional

In [ ]:
# ==============================================================================
# 3.8. COMPILAÇÃO GLOBAL, TESTE DE DIEBOLD-MARIANO E IMPORTÂNCIAS DOS 5 MODELOS
# ==============================================================================
def diebold_mariano_test(y_real, pred1, pred2, crit='MAE'):
    e1 = y_real - pred1
    e2 = y_real - pred2
    d = (np.abs(e1) - np.abs(e2)) if crit == 'MAE' else ((e1 ** 2) - (e2 ** 2))
    mean_d = np.mean(d)
    var_d  = np.var(d, ddof=1) / len(d)
    if var_d == 0:
        return 0.0, 1.0
    dm_stat = mean_d / np.sqrt(var_d)
    p_val   = 2 * (1 - stats.norm.cdf(np.abs(dm_stat)))
    return dm_stat, p_val

nomes_modelos = ['1_OLS_Stepwise', '2_ElasticNet', '3_DecisionTree_CART', '4_RandomForest', '5_XGBoost']
cols_imp_todos = ['OLS_Imp_%', 'ElasticNet_Imp_%', 'CART_Imp_%', 'RF_SHAP_Imp_%', 'XGB_SHAP_Imp_%']

tabelas_desempenho_dict = {}
tabelas_dm_dict         = {}
tabelas_imp_agrup_dict  = {}

for dv in LISTA_DVS:
    # 1. Tabela de Desempenho dos 5 Modelos
    df_desemp = pd.DataFrame(list(resultados_metricas[dv].values())).sort_values(by='MAE_Teste')
    tabelas_desempenho_dict[dv] = df_desemp

    # 2. Tabela do Teste de Diebold-Mariano (todos os pares entre os 5 modelos)
    linhas_dm = []
    y_real_val = y_test_dict[dv].values
    for i in range(len(nomes_modelos)):
        for j in range(i + 1, len(nomes_modelos)):
            m1, m2 = nomes_modelos[i], nomes_modelos[j]
            dm_mae, p_mae = diebold_mariano_test(y_real_val, previsoes_teste[dv][m1], previsoes_teste[dv][m2], crit='MAE')
            dm_mse, p_mse = diebold_mariano_test(y_real_val, previsoes_teste[dv][m1], previsoes_teste[dv][m2], crit='MSE')
            melhor = m2 if dm_mae > 0 else m1
            linhas_dm.append({
                'Variavel_Dependente': dv, 'Modelo_1': m1, 'Modelo_2': m2, 'Modelo_Menor_Erro': melhor,
                'Estatistica_DM (MAE)': round(dm_mae, 4), 'p_value (MAE)': round(p_mae, 4),
                'Estatistica_DM (RMSE)': round(dm_mse, 4), 'p_value (RMSE)': round(p_mse, 4),
                'Significativo_5%': 'Sim' if (p_mae < 0.05 or p_mse < 0.05) else 'Não'
            })
    tabelas_dm_dict[dv] = pd.DataFrame(linhas_dm)

    # 3. Tabela de Importância Agrupada Integrando os 5 Modelos (Lineares + Árvores)
    df_imp_lin = importancia_detalhada_inter[dv].groupby('Variavel_Origem', as_index=False)[['OLS_Imp_%', 'ElasticNet_Imp_%']].sum()
    df_imp_arv = importancia_detalhada_base[dv].groupby('Variavel_Origem', as_index=False)[['CART_Imp_%', 'RF_SHAP_Imp_%', 'XGB_SHAP_Imp_%']].sum()

    df_imp_merged = pd.merge(df_imp_lin, df_imp_arv, on='Variavel_Origem', how='outer').fillna(0.0)
    df_imp_merged['Media_Global_%'] = df_imp_merged[cols_imp_todos].mean(axis=1)
    df_imp_merged.insert(0, 'Variavel_Dependente', dv)
    tabelas_imp_agrup_dict[dv] = df_imp_merged.sort_values(by='Media_Global_%', ascending=False).round(2)

    print(f"\n==================================================================")
    print(f"RESUMO COMPARATIVO PARA A VARIÁVEL DEPENDENTE: {dv}")
    print(f"==================================================================")
    display(tabelas_desempenho_dict[dv])
    print(f"\nTop 10 Variáveis / Interações mais importantes nos 5 Modelos ({dv}):")
    display(tabelas_imp_agrup_dict[dv].head(10))

df_diagnostico_computacional = pd.DataFrame(registo_computacional)
print("\n=== DIAGNÓSTICO DE CUSTO COMPUTACIONAL DOS 5 MODELOS ===")
display(df_diagnostico_computacional)


RESUMO COMPARATIVO PARA A VARIÁVEL DEPENDENTE: Mosteller_3


,Variavel_Dependente,Modelo,MAE_Treino,RMSE_Treino,R2_Treino,MAE_Teste,RMSE_Teste,MAPE_Teste (%),R2_Teste,Tempo_Treino (s),Hiperparametros
4,Mosteller_3,5_XGBoost,0.1495,0.2276,0.9734,0.3499,0.5282,15.14,0.8344,2.5227,"{'subsample': 1.0, 'n_estimators': 100, 'min_c..."
2,Mosteller_3,3_DecisionTree_CART,0.5164,0.7084,0.7419,0.5826,0.8205,26.33,0.6005,1.7992,"{'max_depth': None, 'min_samples_leaf': 0.05, ..."
3,Mosteller_3,4_RandomForest,0.5890,0.7899,0.6790,0.6078,0.8206,27.59,0.6004,8.2208,"{'n_estimators': 100, 'min_samples_leaf': 0.05..."
1,Mosteller_3,2_ElasticNet,0.6264,0.8590,0.6205,0.6261,0.8988,26.97,0.5206,38.9101,"alpha=0.0569, l1_ratio=0.2 (71/206 vars ativas)"
0,Mosteller_3,1_OLS_Stepwise,0.6658,0.9008,0.5826,0.6584,0.8970,29.01,0.5226,12.2750,"Stepwise (21 iterações, 17/206 vars selecionadas)"



Top 10 Variáveis / Interações mais importantes nos 5 Modelos (Mosteller_3):


,Variavel_Dependente,Variavel_Origem,OLS_Imp_%,ElasticNet_Imp_%,CART_Imp_%,RF_SHAP_Imp_%,XGB_SHAP_Imp_%,Media_Global_%
16,Mosteller_3,dias_desde_ulti_eleicao,0.00,5.39,49.04,26.20,28.070000,21.74
4,Mosteller_3,Interacao_Instituto_x_Quant,36.92,41.52,0.00,0.00,0.000000,15.69
1,Mosteller_3,CDU,0.00,3.60,20.78,11.22,16.700001,10.46
5,Mosteller_3,Interacao_Modo_x_Quant,20.98,25.10,0.00,0.00,0.000000,9.22
8,Mosteller_3,PAN,0.00,0.97,15.08,16.65,11.070000,8.75
15,Mosteller_3,dias_ate_prox_eleicao,2.46,0.00,1.92,20.91,5.770000,6.21
0,Mosteller_3,BE,5.77,6.72,1.41,4.54,7.750000,5.24
9,Mosteller_3,PS,3.78,3.38,2.30,1.26,7.270000,3.60
10,Mosteller_3,PSD-CDS,0.00,0.00,8.28,2.27,5.640000,3.24
2,Mosteller_3,CH,9.06,2.12,0.00,0.71,1.910000,2.76



RESUMO COMPARATIVO PARA A VARIÁVEL DEPENDENTE: MAPE_Sondagem


,Variavel_Dependente,Modelo,MAE_Treino,RMSE_Treino,R2_Treino,MAE_Teste,RMSE_Teste,MAPE_Teste (%),R2_Teste,Tempo_Treino (s),Hiperparametros
4,MAPE_Sondagem,5_XGBoost,1.3244,2.0808,0.9772,3.6901,5.4032,16.83,0.8669,2.4478,"{'subsample': 0.8, 'n_estimators': 150, 'min_c..."
1,MAPE_Sondagem,2_ElasticNet,4.8167,6.1786,0.7994,5.7990,7.5461,27.94,0.7404,57.6757,"alpha=0.0720, l1_ratio=1.0 (69/206 vars ativas)"
0,MAPE_Sondagem,1_OLS_Stepwise,5.3738,6.9228,0.7481,5.8368,7.8330,28.44,0.7203,15.5931,"Stepwise (23 iterações, 21/206 vars selecionadas)"
2,MAPE_Sondagem,3_DecisionTree_CART,5.7946,8.1689,0.6493,6.3963,8.8978,29.26,0.6391,0.3315,"{'max_depth': None, 'min_samples_leaf': 0.05, ..."
3,MAPE_Sondagem,4_RandomForest,5.5853,7.8750,0.6741,6.8091,9.2948,33.37,0.6062,8.2109,"{'n_estimators': 150, 'min_samples_leaf': 0.05..."


... (4 outputs left out)


Top 10 Variáveis / Interações mais importantes nos 5 Modelos (Medida_B):


,Variavel_Dependente,Variavel_Origem,OLS_Imp_%,ElasticNet_Imp_%,CART_Imp_%,RF_SHAP_Imp_%,XGB_SHAP_Imp_%,Media_Global_%
16,Medida_B,dias_desde_ulti_eleicao,8.07,19.96,39.08,37.98,24.67,25.95
4,Medida_B,Interacao_Instituto_x_Quant,35.29,30.56,0.00,0.00,0.00,13.17
0,Medida_B,BE,4.67,10.86,21.74,7.92,10.56,11.15
8,Medida_B,PAN,3.55,9.86,16.33,8.26,8.33,9.27
15,Medida_B,dias_ate_prox_eleicao,2.74,0.00,8.94,15.36,9.95,7.40
5,Medida_B,Interacao_Modo_x_Quant,26.62,3.10,0.00,0.00,0.00,5.94
10,Medida_B,PSD-CDS,4.19,7.14,0.33,11.26,6.14,5.81
1,Medida_B,CDU,0.00,4.54,3.49,2.90,5.73,3.33
6,Medida_B,L,9.54,0.00,0.00,2.08,4.87,3.30
14,Medida_B,demissao_gov,3.43,12.47,0.00,0.32,0.22,3.29



=== DIAGNÓSTICO DE CUSTO COMPUTACIONAL DOS 5 MODELOS ===


,Variavel_Dependente,Modelo,Tempo_Execucao (s),Hardware_CPU,RAM_Total,Detalhes_Processamento
0,Mosteller_3,1_OLS_Stepwise,12.2750,x86_64 (1 núcleos físicos / 2 lógicos),12.67 GB,"Stepwise (21 iterações, 17/206 vars selecionadas)"
1,MAPE_Sondagem,1_OLS_Stepwise,15.5931,x86_64 (1 núcleos físicos / 2 lógicos),12.67 GB,"Stepwise (23 iterações, 21/206 vars selecionadas)"
2,Medida_B,1_OLS_Stepwise,17.9306,x86_64 (1 núcleos físicos / 2 lógicos),12.67 GB,"Stepwise (29 iterações, 23/206 vars selecionadas)"
3,Mosteller_3,2_ElasticNet,38.9101,x86_64 (1 núcleos físicos / 2 lógicos),12.67 GB,"alpha=0.0569, l1_ratio=0.2 (71/206 vars ativas)"
4,MAPE_Sondagem,2_ElasticNet,57.6757,x86_64 (1 núcleos físicos / 2 lógicos),12.67 GB,"alpha=0.0720, l1_ratio=1.0 (69/206 vars ativas)"
5,Medida_B,2_ElasticNet,12.1585,x86_64 (1 núcleos físicos / 2 lógicos),12.67 GB,"alpha=0.0043, l1_ratio=1.0 (22/206 vars ativas)"
6,Mosteller_3,3_DecisionTree_CART,1.7992,x86_64 (1 núcleos físicos / 2 lógicos),12.67 GB,"{'max_depth': None, 'min_samples_leaf': 0.05, ..."
7,MAPE_Sondagem,3_DecisionTree_CART,0.3315,x86_64 (1 núcleos físicos / 2 lógicos),12.67 GB,"{'max_depth': None, 'min_samples_leaf': 0.05, ..."
8,Medida_B,3_DecisionTree_CART,0.3340,x86_64 (1 núcleos físicos / 2 lógicos),12.67 GB,"{'max_depth': None, 'min_samples_leaf': 0.05, ..."
9,Mosteller_3,4_RandomForest,8.2208,x86_64 (1 núcleos físicos / 2 lógicos),12.67 GB,"{'n_estimators': 100, 'min_samples_leaf': 0.05..."


## 3.9. Exportação para Excel Único com Separadores por Variável Dependente + Regras da Árvore + Diagnóstico

In [ ]:
# ==============================================================================
# 3.9. EXPORTAÇÃO PARA EXCEL ÚNICO COM TODOS OS BLOCOS E REGRAS DAS ÁRVORES
# ==============================================================================
caminho_excel_unico = '/content/drive/MyDrive/A Máquina de Sondagens/Resultados_Modelos_Cap4_3DVs_Atualizado.xlsx'

lista_regras_todas = []
for dv in LISTA_DVS:
    lista_regras_todas.extend(regras_modelos_dict[dv])
df_regras_export = pd.DataFrame(lista_regras_todas)

with pd.ExcelWriter(caminho_excel_unico) as writer:
    # 1. Separador dedicado a cada Variável Dependente ('Mosteller_3', 'MAPE_Sondagem', 'Medida_B')
    for dv in LISTA_DVS:
        row_cursor = 0

        # Bloco A: Desempenho dos 5 Modelos
        t_desemp = tabelas_desempenho_dict[dv]
        t_desemp.to_excel(writer, sheet_name=dv, startrow=row_cursor, index=False)
        row_cursor += len(t_desemp) + 3

        # Bloco B: Teste de Diebold-Mariano
        t_dm = tabelas_dm_dict[dv]
        t_dm.to_excel(writer, sheet_name=dv, startrow=row_cursor, index=False)
        row_cursor += len(t_dm) + 3

        # Bloco C: Importância Agrupada nos 5 Modelos (%)
        t_imp_ag = tabelas_imp_agrup_dict[dv]
        t_imp_ag.to_excel(writer, sheet_name=dv, startrow=row_cursor, index=False)
        row_cursor += len(t_imp_ag) + 3

        # Bloco D: Variáveis e Interações Selecionadas pelo Modelo 1 (1_OLS_Stepwise)
        t_ols = tabelas_ols_dict[dv]
        t_ols.to_excel(writer, sheet_name=dv, startrow=row_cursor, index=False)
        row_cursor += len(t_ols) + 3

        # Bloco E: Variáveis e Interações Ativas no Modelo 2 (2_ElasticNet)
        t_enet = tabelas_enet_dict[dv]
        t_enet.to_excel(writer, sheet_name=dv, startrow=row_cursor, index=False)
        row_cursor += len(t_enet) + 3

        # Bloco F: Importância Detalhada das 30 Variáveis Base nas Árvores (Modelos 3, 4 e 5)
        t_arv_det = importancia_detalhada_base[dv].round(4)
        t_arv_det.to_excel(writer, sheet_name=dv, startrow=row_cursor, index=False)

    # 2. Separador com o Resumo Global das 3 Variáveis Dependentes
    pd.concat(list(tabelas_desempenho_dict.values()), ignore_index=True).to_excel(
        writer, sheet_name='Resumo_Global_3DVs', index=False
    )

    # 3. Separador com as Regras de Decisão Extraídas dos Modelos 3, 4 e 5
    df_regras_export.to_excel(writer, sheet_name='Regras_Modelos_3_4_5', index=False)

    # 4. Separador com o Custo Computacional dos 5 Modelos
    df_diagnostico_computacional.to_excel(writer, sheet_name='Custo_Computacional', index=False)

print("Ficheiro Excel completo exportado com sucesso para:", caminho_excel_unico)
print("Separadores criados:", LISTA_DVS + ['Resumo_Global_3DVs', 'Regras_Modelos_3_4_5', 'Custo_Computacional'])

Ficheiro Excel completo exportado com sucesso para: /content/drive/MyDrive/A Máquina de Sondagens/Resultados_Modelos_Cap4_3DVs_Atualizado.xlsx
Separadores criados: ['Mosteller_3', 'MAPE_Sondagem', 'Medida_B', 'Resumo_Global_3DVs', 'Regras_Modelos_3_4_5', 'Custo_Computacional']
